### 1:

In [10]:
import sqlite3 
import pandas as pd
db_ref = 'AdventureWorks.db'

In [2]:


queryQ1 = '''

SELECT MIN(p.ListPrice) AS min_price, MAX(p.ListPrice) AS max_price, MAX(p.ListPrice) - MIN(p.ListPrice) as price_diff, pc.ParentProductCategoryID, pc.Name, COUNT(*) AS 'type quantity'
FROM productcategory as pc
JOIN product as p
ON p.ProductCategoryID=pc.ProductCategoryID
GROUP BY pc.ParentProductCategoryID
'''

with sqlite3.connect(db_ref) as conn:
        price_ranges = pd.read_sql_query(queryQ1, conn)

price_ranges.head()

,min_price,max_price,price_diff,ParentProductCategoryID,Name,type quantity
0,539.99,3578.27,3038.28,1,Road Bikes,97
1,20.24,1431.50,1411.26,2,Road Frames,134
2,8.99,89.99,81.00,3,Bib-Shorts,35
3,2.29,159.00,156.71,4,Bike Stands,29


In [7]:
sql_query = '''
    SELECT ParentProductCategoryID,
        (SELECT name
        FROM ProductCategory
        WHERE (ProductCategoryID == pc.ParentProductCategoryID)
        ) as Name, MIN(p.ListPrice) AS min_price, MAX(p.ListPrice) AS max_price, MAX(p.ListPrice) - MIN(p.ListPrice) as price_diff, COUNT(*) AS 'type quantity'
    FROM ProductCategory AS pc
        JOIN Product AS p
        ON pc.ProductCategoryID = p.ProductCategoryID
    GROUP BY pc.ParentProductCategoryID
            '''

with sqlite3.connect(db_ref) as conn:
    df = pd.read_sql_query(sql_query, conn)
df.head()

,ParentProductCategoryID,Name,min_price,max_price,price_diff,type quantity
0,1,Bikes,539.99,3578.27,3038.28,97
1,2,Components,20.24,1431.50,1411.26,134
2,3,Clothing,8.99,89.99,81.00,35
3,4,Accessories,2.29,159.00,156.71,29


### 2. Price Ranges by Subcategory

In [4]:
from sympy import product

queryQ1 = '''

SELECT MIN(p.ListPrice) AS min_price, MAX(p.ListPrice) AS max_price, MAX(p.ListPrice) - MIN(p.ListPrice) as price_diff, pc.name, COUNT(*) AS 'type quantity'
FROM product as p
JOIN productcategory as pc
ON p.ProductCategoryID=pc.ProductCategoryID
GROUP BY p.ProductCategoryID
'''

with sqlite3.connect(db_ref) as conn:
        price_ranges = pd.read_sql_query(queryQ1, conn)

price_ranges.head()

,min_price,max_price,price_diff,Name,type quantity
0,539.99,3399.99,2860.00,Mountain Bikes,32
1,539.99,3578.27,3038.28,Road Bikes,43
2,742.35,2384.07,1641.72,Touring Bikes,22
3,44.54,120.27,75.73,Handlebars,8
4,53.99,121.49,67.50,Bottom Brackets,3


In [34]:
queryQ2 = '''
SELECT *
FROM product
'''
with sqlite3.connect(db_ref) as conn:
    price_basket = pd.read_sql_query(queryQ2, conn)


price_basket["Price Bracket"] = pd.cut(
    price_basket["ListPrice"],
    bins=[0, 50, 100, 500, 1000, float("inf")],
    labels=["Under $50","Under $100", "Under $500", "Under $1000", "Over $1000"],
    right=False
)

price_basket.sample(10)

,ProductID,Name,ProductNumber,Color,StandardCost,ListPrice,Size,Weight,ProductCategoryID,ProductModelID,SellStartDate,SellEndDate,DiscontinuedDate,ThumbNailPhoto,ThumbnailPhotoFileName,rowguid,ModifiedDate,Price Bracket
213,918,"LL Mountain Frame - Silver, 44",FR-M21S-44,Silver,144.5938,264.050,44,1342.63,16,8,2007-07-01 00:00:00.000,,,47494638396150003100F7000000000080000000800080...,no_image_available_small.gif,393A4D00-7428-41F0-A48A-AF26B00E9A9C,2008-03-11 10:01:36.827,Under $500
60,765,"Road-650 Black, 58",BK-R50B-58,Black,486.7066,782.990,58,8976.55,6,30,2005-07-01 00:00:00.000,2007-06-30 00:00:00.000,,47494638396150003100F70000EEEDF8F2F1F2EDEDF5E0...,superlight_black_small.gif,74645B12-3CE9-49A6-BD96-2CD814B37420,2008-03-11 10:01:36.827,Under $1000
207,912,ML Road Seat/Saddle,SE-R908,,17.3782,39.140,,,19,83,2007-07-01 00:00:00.000,,,47494638396150003100F7000000000080000000800080...,no_image_available_small.gif,49845AFE-A8CC-4354-A5D4-09D35BF7FB9E,2008-03-11 10:01:36.827,Under $50
150,855,"Men's Bib-Shorts, S",SB-M891-S,Multi,37.1209,89.990,S,,22,12,2006-07-01 00:00:00.000,2007-06-30 00:00:00.000,,47494638396150003100F7000000000080000000800080...,no_image_available_small.gif,9F60AF1E-4C11-4E90-BAEA-48E834E8B4C2,2008-03-11 10:01:36.827,Under $100
89,794,"Road-250 Black, 48",BK-R89B-48,Black,1554.9479,2443.350,48,6862.82,6,26,2006-07-01 00:00:00.000,,,47494638396150003100F70000E3E3FCB8BBC8ACADB055...,racer_black_small.gif,9D165DDF-8F5D-41C7-9BB8-13F41A3D1F62,2008-03-11 10:01:36.827,Over $1000
47,752,"Road-150 Red, 52",BK-R93R-52,Red,2171.2942,3578.270,52,6540.77,6,25,2005-07-01 00:00:00.000,2006-06-30 00:00:00.000,,47494638396150003100F70000920407C6BCC339343A58...,superlight_red_small.gif,5E085BA0-3CD5-487F-85BB-79ED1C701F23,2008-03-11 10:01:36.827,Over $1000
293,998,"Road-750 Black, 48",BK-R19B-48,Black,343.6496,539.990,48,9130.77,6,31,2007-07-01 00:00:00.000,,,47494638396150003100F70000CDC9CAC3C2C2F0F3EBDA...,roadster_black_small.gif,3DE9A212-1D49-40B6-B10A-F564D981DBDE,2008-03-11 10:01:36.827,Under $1000
262,967,"Touring-1000 Blue, 50",BK-T79U-50,Blue,1481.9379,2384.070,50,11530.26,7,34,2007-07-01 00:00:00.000,,,47494638396150003100F700007477789EA4AA3B41442E...,julianax_r_02_blue_small.gif,68C0A818-2985-46EB-8255-0FB70919FA24,2008-03-11 10:01:36.827,Over $1000
227,932,ML Road Tire,TI-R628,,9.3463,24.990,,,41,89,2007-07-01 00:00:00.000,,,47494638396150003100F7000000000080000000800080...,no_image_available_small.gif,D1016CC5-C12B-4F05-915C-70FA062E4A64,2008-03-11 10:01:36.827,Under $50
120,825,HL Mountain Rear Wheel,RW-M928,Black,145.2835,327.215,,,21,125,2006-07-01 00:00:00.000,2007-06-30 00:00:00.000,,47494638396133003200F700009D9E9EFEFCFCF23735FE...,wheel_small.gif,35D02EDC-1120-41FB-B5DF-8655A93B3353,2008-03-11 10:01:36.827,Under $500


In [35]:
price_basket['Price Bracket'].value_counts()

Price Bracket
Over $1000     86
Under $500     69
Under $50      53
Under $1000    50
Under $100     37
Name: count, dtype: int64

In [27]:
queryQ2 = '''
SELECT
CASE
WHEN ListPrice < 100 THEN 'Cheap'
WHEN ListPrice BETWEEN 1000 AND 2000 THEN 'Medium'
ELSE 'Expensive'
END AS Price_Basket,
COUNT(*) AS Product_Count
FROM product
GROUP BY Price_Basket;
'''
with sqlite3.connect(db_ref) as conn:
    price_basket = pd.read_sql_query(queryQ2, conn)

price_basket.head()



,Price_Basket,Product_Count
0,Cheap,90
1,Expensive,154
2,Medium,51


## HR Questions!!!

### Question 1

In [ ]:
Query3 = '''

FROM 

'''